### 04. Спорт и диета: работает ли сочетание?

Влияет ли сочетание физической активности и качества диеты на оценку за экзамен? Отличаются ли группы с разными профилями по среднему баллу и доле высоких оценок? Есть ли взаимодействие между спортом и диетой — то есть помогает ли спорт сильнее при одной диете, чем при другой?

In [1]:
import sys
import os
from pathlib import Path

project_root = Path(os.getcwd()).parent 
sys.path.insert(0, str(project_root))

from src.preparation import load_df
df = load_df()

In [13]:
import matplotlib.pyplot as plt 
import numpy as np 

plt.rcParams['font.family'] = 'Arial'

df_02_04 = df.copy()

df_02_04['lifestyle_profile'] = np.select(
    [
        (df_02_04.exercise_frequency >= 4) & (df_02_04.diet_quality == 'Good'), 
        (df_02_04.exercise_frequency < 4) & (df_02_04.diet_quality == 'Good'), 
        (df_02_04.exercise_frequency >= 4) & (df_02_04.diet_quality == 'Fair'), 
        (df_02_04.exercise_frequency < 4) & (df_02_04.diet_quality == 'Fair'),
        (df_02_04.exercise_frequency >= 4) & (df_02_04.diet_quality == 'Poor'),
        # (df_02_04.exercise_frequency < 4) & (df_02_04.diet_quality == 'Poor')
    ], 
    ['A', 'B', 'C', 'D', 'E'], default='F'
)

res_02_04 = df_02_04.groupby('lifestyle_profile').agg(
    students_qty=('student_id', 'count'),
    avg_exam_score=('exam_score', 'mean'), 
    std_exam_score=('exam_score', 'std'), 
    exam_score_80_or_more_pct=('exam_score', lambda i: (i >= 80).sum() / i.count() * 100), 
    avg_mental_health=('mental_health_rating', 'mean')
)

display(res_02_04.style.format('{:.1f}').background_gradient(subset=['avg_exam_score', 'exam_score_80_or_more_pct'], cmap='RdYlGn'))

,students_qty,avg_exam_score,std_exam_score,exam_score_80_or_more_pct,avg_mental_health
lifestyle_profile,,,,,
A,173.0,71.3,17.6,33.5,5.6
B,205.0,67.7,16.5,23.9,5.7
C,179.0,74.0,16.1,33.5,5.0
D,258.0,67.9,16.6,25.2,5.4
E,83.0,69.8,17.7,28.9,5.7
F,102.0,66.7,16.5,20.6,5.5


#### Расшифровка профилей:

| Профиль | Спорт (раз/нед.) | Диета   |
| ------- | ---------------- | ------- |
| A       | ≥ 4              | Хорошая |
| B       | < 4              | Хорошая |
| C       | ≥ 4              | Средняя |
| D       | < 4              | Средняя |
| E       | ≥ 4              | Плохая    |
| F       | < 4              | Плохая    |

**_Как читать:_**

- **первая пара (A–B)** — студенты с хорошей диетой;
- **вторая пара (C–D)** — студенты со средней диетой;
- **третья пара (E–F)** — студенты с хорошей диетой.

_Внутри каждой пары верхняя группа занимается спортом ≥ 4 раз в неделю, нижняя — меньше._


#### 04. Выводы

**1. Спорт связан с успеваемостью.**
Во всех трёх парах группа со спортом ≥ 4 раз/нед показывает более высокий средний балл, чем парная с < 4:
- Good: A (71.3) vs B (67.7) → +3.6
- Fair: C (74.0) vs D (67.9) → +6.1
- Poor: E (69.8) vs F (66.7) → +3.1

**2. Диета не показывает линейной связи.**
При спорте ≥ 4: A (71.3) vs C (74.0) vs E (69.8) — разница 4.2 балла.
При спорте < 4: B (67.7) vs D (67.9) vs F (66.7) — разница 1.2 балла.
Различия малы относительно std ≈ 16–17 и не позволяют говорить о значимом эффекте диеты.

**3. Interaction слабый.**
Эффект спорта сильнее при Fair диете (+6.1), чем при Good (+3.6) и Poor (+3.1). Разница 2–3 балла находится на грани статистической значимости.

**4. Контрастные группы.**  
C (спорт ≥ 4 + Fair) — лидер: 74.0 среднего, 33.5% > 80.  
F (спорт < 4 + Poor) — аутсайдер: 66.7 среднего, 20.6% > 80.  
Разрыв — 7.3 балла и 12.9 п.п.  

**5. Mental health не отличает профили.** Средний уровень ментального здоровья во всех шести группах находится в диапазоне **5.0–5.7**. Различия в пределах **0.7 балла** по 10-балльной шкале не позволяют говорить о связи между образом жизни и ментальным здоровьем в данной выборке. Примечательно, что группа **C** — лидер по успеваемости (74.0) — показывает **самый низкий** уровень mental health (5.0), а группа **E** — средняя по успеваемости (69.8) — **самый высокий** (5.7). Это указывает на отсутствие простой линейной связи между ментальным здоровьем и академическим результатом в разрезе профилей образа жизни.

**Ограничения.** Столбец diet_quality — самоотчёт. Направление связи «спорт → оценка» не установлено: возможно, студенты с высокой успеваемостью изначально склонны к физической активности. 